# Trading Code Generator — Multi-Model Comparison

This notebook implements an LLM-powered **trading code generator** that:
- Generates Python trading strategies to buy/sell equities in a **simulated environment**
- Compares multiple models (GPT, Claude, Gemini, Grok, etc.) side by side
- Ranks models by performance (profit %, final value, number of trades)
- Provides a **Gradio UI** for interactive comparison

### Setup
Ensure your `.env` file (in project root) contains API keys for the models you want to use:
- `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`, `GROK_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`
- For local models, run Ollama with `ollama run llama3.2` (or qwen2.5-coder).

In [ ]:
# Imports
import os
import traceback
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import numpy as np

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# Connect to client libraries
openai_client = OpenAI()
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url) if google_api_key else None
grok = OpenAI(api_key=grok_api_key, base_url=grok_url) if grok_api_key else None
groq = OpenAI(api_key=groq_api_key, base_url=groq_url) if groq_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url) if openrouter_api_key else None

print("API clients initialized.")

In [ ]:
# Models and clients - customize based on your API keys
# Add/remove models as needed; use openrouter for many open models
models_config = []
clients = {}

if openai_api_key:
    models_config.extend(["gpt-4o", "gpt-4o-mini"])
    clients.update({"gpt-4o": openai_client, "gpt-4o-mini": openai_client})
if anthropic_api_key:
    models_config.extend(["claude-sonnet-4-20250514", "claude-3-5-haiku-20241022"])
    clients.update({"claude-sonnet-4-20250514": anthropic, "claude-3-5-haiku-20241022": anthropic})
if google_api_key:
    models_config.extend(["gemini-2.0-flash", "gemini-1.5-pro"])
    clients.update({"gemini-2.0-flash": gemini, "gemini-1.5-pro": gemini})
if grok_api_key:
    models_config.extend(["grok-3"])
    clients.update({"grok-3": grok})
if groq_api_key:
    models_config.extend(["llama-3.3-70b-versatile"])
    clients.update({"llama-3.3-70b-versatile": groq})
if openrouter_api_key:
    models_config.extend(["openai/gpt-4o-mini", "anthropic/claude-3.5-haiku"])
    clients.update({"openai/gpt-4o-mini": openrouter, "anthropic/claude-3.5-haiku": openrouter})

# Fallback: if no APIs, use Ollama (local)
if not models_config:
    models_config = ["llama3.2", "qwen2.5-coder"]
    clients = {"llama3.2": ollama, "qwen2.5-coder": ollama}
else:
    models_config.extend(["llama3.2", "qwen2.5-coder"])
    clients.update({"llama3.2": ollama, "qwen2.5-coder": ollama})

print("Available models:", models_config)

## Simulated Trading Environment

A simple backtester that provides `buy`, `sell`, and price data. Generated code runs in a sandbox with access to this simulator.

In [ ]:
def generate_price_series(seed=42, days=100, start_price=100.0, volatility=0.02):
    """Generate realistic-looking price series (random walk with drift)."""
    np.random.seed(seed)
    returns = np.random.normal(0.0002, volatility, days)
    prices = start_price * np.exp(np.cumsum(returns))
    return list(prices)

class TradingSimulator:
    """Simulated trading env: buy/sell at daily closes. Code uses sim.buy(day, qty), sim.sell(day, qty)."""
    def __init__(self, prices, initial_cash=100_000.0):
        self.prices = prices
        self.cash = initial_cash
        self.shares = 0.0
        self.trades = []  # (day, "buy"/"sell", qty, price)
        self.initial_cash = initial_cash

    def buy(self, day, shares):
        if day < 0 or day >= len(self.prices) or shares <= 0:
            return
        price = self.prices[day]
        cost = shares * price
        if cost <= self.cash:
            self.cash -= cost
            self.shares += shares
            self.trades.append((day, "buy", shares, price))

    def sell(self, day, shares):
        if day < 0 or day >= len(self.prices) or shares <= 0:
            return
        sell_qty = min(shares, self.shares)
        if sell_qty > 0:
            price = self.prices[day]
            self.cash += sell_qty * price
            self.shares -= sell_qty
            self.trades.append((day, "sell", sell_qty, price))

    def portfolio_value(self, day=None):
        day = day if day is not None else len(self.prices) - 1
        day = min(day, len(self.prices) - 1)
        return self.cash + self.shares * self.prices[day]

    def final_value(self):
        return self.portfolio_value(len(self.prices) - 1)

# Default price series for evaluation
DEFAULT_PRICES = generate_price_series(seed=42, days=100)

## Code Generation Prompts

The LLM receives a strategy description and must output Python code that uses the `sim` object.

In [ ]:
SYSTEM_PROMPT = """You are a trading algorithm developer. Your task is to write Python code that implements a trading strategy in a simulated environment.

You have access to a variable `sim` which is a TradingSimulator with:
- sim.prices: list of daily closing prices (index 0 = day 0, 1 = day 1, ...)
- sim.buy(day, shares): buy `shares` at the closing price of `day`
- sim.sell(day, shares): sell `shares` at the closing price of `day`
- sim.cash: current cash balance
- sim.shares: current shares held
- sim.portfolio_value(day): cash + shares * price at `day`

Your code will be executed with `sim` already created. Loop through days (0 to len(sim.prices)-1) and call sim.buy() or sim.sell() based on your strategy.
Do NOT redefine sim. Do NOT use print statements. Respond ONLY with valid Python code, no markdown or explanation."""

def user_prompt(strategy_description):
    return f"""Write Python trading code for this strategy:

{strategy_description}

Use the `sim` object (TradingSimulator) provided. Iterate over days 0 to len(sim.prices)-1 and call sim.buy(day, qty) or sim.sell(day, qty).
Respond ONLY with Python code."""

In [ ]:
def generate_code(model, strategy_description):
    """Generate trading code using the specified model."""
    if model not in clients or clients[model] is None:
        return None, f"Model {model} not available"
    client = clients[model]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(strategy_description)}
    ]
    try:
        response = client.chat.completions.create(model=model, messages=messages)
        code = response.choices[0].message.content
        # Strip markdown code blocks
        for delim in ["```python", "```"]:
            code = code.replace(delim, "")
        code = code.strip()
        return code, None
    except Exception as e:
        return None, str(e)

In [ ]:
def run_strategy(code, prices=None):
    """Execute generated code in sandbox. Returns dict with profit_pct, final_value, error, trades, sharpe (approx)."""
    prices = prices or DEFAULT_PRICES
    sim = TradingSimulator(prices)
    safe_builtins = {
        "len": len, "range": range, "enumerate": enumerate, "zip": zip,
        "min": min, "max": max, "sum": sum, "abs": abs, "round": round,
        "list": list, "float": float, "int": int, "True": True, "False": False,
    }
    ns = {"sim": sim, "__builtins__": safe_builtins}
    try:
        exec(code, ns)
        final = sim.final_value()
        ret = (final - sim.initial_cash) / sim.initial_cash * 100
        # Simple Sharpe proxy: return / (1 + std of daily returns) if we had them
        n_trades = len(sim.trades)
        return {
            "final_value": final,
            "profit_pct": ret,
            "trades": n_trades,
            "error": None,
            "success": True,
        }
    except Exception as e:
        return {
            "final_value": sim.initial_cash,
            "profit_pct": 0,
            "trades": 0,
            "error": traceback.format_exc(),
            "success": False,
        }

In [ ]:
def evaluate_all_models(strategy_description, models_to_run=None):
    """Generate code with each model, run simulation, return ranked results."""
    models_to_run = models_to_run or [m for m in models_config if m in clients and clients[m] is not None]
    results = []
    for model in models_to_run:
        code, err = generate_code(model, strategy_description)
        if err:
            results.append({"model": model, "profit_pct": -999, "error": err, "success": False, "code": ""})
            continue
        r = run_strategy(code)
        r["model"] = model
        r["code"] = code
        results.append(r)
    # Rank by profit_pct (descending)
    results.sort(key=lambda x: x["profit_pct"], reverse=True)
    return results

In [ ]:
# Quick test: run a simple buy-hold strategy manually
sim = TradingSimulator(DEFAULT_PRICES)
n = len(sim.prices)
# Buy 500 shares on day 0, sell on last day
sim.buy(0, 500)
sim.sell(n - 1, 500)
print(f"Buy-hold test: Initial $100k -> Final ${sim.final_value():,.0f} ({sim.final_value()/100000*100-100:.1f}%)")

## Gradio UI

Interactive interface to:
1. Enter a strategy description
2. Select a model and generate code
3. Run the strategy and see results
4. **Compare All Models** — run all models and see the leaderboard (best model wins!)

In [ ]:
# Styling for the leaderboard
CSS = """
.leaderboard-box { font-family: monospace; padding: 16px; background: #1a1d23; border-radius: 10px; }
.rank-1 { color: #ffd700 !important; font-weight: bold; font-size: 1.1em; }
.rank-2 { color: #c0c0c0 !important; }
.rank-3 { color: #cd7f32 !important; }
"""

In [ ]:
DEFAULT_STRATEGY = """Buy and hold: invest 50% of cash on day 0, then sell everything on the last day."""

def format_leaderboard(results):
    lines = ["## 🏆 Model Leaderboard (ranked by profit %)\n"]
    for i, r in enumerate(results, 1):
        medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}."
        err = f" — Error: {r.get('error', '')[:80]}..." if not r.get("success", True) else ""
        profit = r.get("profit_pct", 0)
        trades = r.get("trades", 0)
        lines.append(f"**{medal} {r['model']}** — Profit: {profit:.2f}% | Trades: {trades}{err}")
    return "\n".join(lines)

In [ ]:
def ui_generate(model, strategy):
    code, err = generate_code(model, strategy)
    if err:
        return code or "", f"Error: {err}"
    r = run_strategy(code)
    if r["error"]:
        return code, f"Runtime error:\n{r['error'][:500]}"
    return code, f"Profit: {r['profit_pct']:.2f}% | Trades: {r['trades']} | Final: ${r['final_value']:,.0f}"

def ui_compare_all(strategy):
    results = evaluate_all_models(strategy)
    best = results[0] if results else None
    best_code = best.get("code", "") if best else ""
    leaderboard = format_leaderboard(results)
    winner = best["model"] if best and best.get("success") else "None"
    msg = f"Best model: {winner} (profit: {best.get('profit_pct', 0):.2f}%)" if best else "No results"
    return best_code, leaderboard, msg

In [ ]:
with gr.Blocks(css=CSS, title="Trading Code Generator — Multi-Model Comparison") as ui:
    gr.Markdown("# 📈 Trading Code Generator — Compare LLM Models")
    gr.Markdown("Generate trading strategies with different models and see which produces the best code.")

    with gr.Row():
        with gr.Column(scale=1):
            strategy = gr.Textbox(
                label="Strategy description",
                value=DEFAULT_STRATEGY,
                placeholder="e.g., Buy when price drops 2% from the previous day, sell when it rises 3%.",
                lines=4
            )
            with gr.Row():
                model_dropdown = gr.Dropdown(models_config, value=models_config[0], label="Model")
                gen_btn = gr.Button("Generate & Run", variant="primary")
            with gr.Row():
                compare_btn = gr.Button("🏆 Compare All Models", variant="secondary")

        with gr.Column(scale=1):
            code_out = gr.Code(label="Generated code", language="python", lines=15)
            result_out = gr.Textbox(label="Result", lines=3)
            leaderboard_out = gr.Markdown(label="Leaderboard")

    gen_btn.click(
        fn=ui_generate,
        inputs=[model_dropdown, strategy],
        outputs=[code_out, result_out]
    )
    compare_btn.click(
        fn=ui_compare_all,
        inputs=[strategy],
        outputs=[code_out, leaderboard_out, result_out]
    )

ui.launch(inbrowser=True)